# 01 — Data Understanding

**Business problem.** Banks approve/decline card transactions in milliseconds. Fraud is ~0.17% of
transactions but each miss costs the full amount plus chargeback fees, while each false alarm costs
customer friction and support time. We need a model that outputs a **fraud probability**, decided by a
**business-optimized threshold**.

**Dataset.** Kaggle *Credit Card Fraud Detection*: 284,807 transactions over 2 days by European
cardholders, 492 frauds (0.172%). `V1–V28` are PCA-anonymized features; only `Time` (seconds since
first transaction), `Amount`, and the target `Class` are raw.

All reusable logic lives in `src/`; this notebook only orchestrates.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import Image, display
from src.utils import load_config, resolve_path

config = load_config()
pd.set_option("display.max_columns", 40)

In [ ]:
from src.data_loader import download_data, load_raw_data

download_data(config)          # no-op if data/raw/creditcard.csv already exists
df = load_raw_data(config)
df.head()

## Structure and types

31 columns: `Time`, `V1–V28`, `Amount`, `Class`. Everything is numeric — no categorical encoding
needed. `V1–V28` are already the output of a PCA transformation done by the dataset publishers for
confidentiality, so they are centered, uncorrelated with each other, and unitless.

In [ ]:
df.info()

In [ ]:
df.describe().T.round(3)

## Data validation

Automated checks (schema, missing values, duplicates, invalid values, target distribution) run via
`src.data_validation`. The report is persisted to `artifacts/validation_report.json` so every run
leaves an audit trail.

Key expected findings:
- **No missing values** anywhere (rare luxury — the dataset is pre-cleaned).
- **1,081 duplicate rows** — dropped *before* splitting so identical rows can never land in both
  train and test (that would leak).
- **Extreme imbalance**: 492 frauds / 284,807 rows ≈ 0.172% → accuracy is meaningless.

In [ ]:
from src.data_validation import run_validation

report = run_validation(df, config)
report